# Example: AI-Driven News Sentiment as a Cobb-Douglas Factor

In this example, Maya extends the Cobb-Douglas rebalancing engine with a news-derived feature, $\text{news}_i$, generated from synthetic headlines and scored by Claude. We layer asymmetric news shocks on the cached 2025 S&P data, recover the per-asset news loadings $\nu_i$ by ordinary least squares, deploy the news-extended $\gamma_i$ in the rebalancing engine, and compare to the SIM-only baseline. We then expose in-sample overfitting by re-fitting the loadings on the test window itself.

> __Learning Objectives:__
>
> * __Generate timestamped synthetic news with asymmetric price feedback:__ Build a news corpus where each item carries a publication day and a planted sentiment, and apply the asymmetric shock $\Delta = -\kappa_{\mathrm{neg}}\max(-s,0) + \kappa_{\mathrm{pos}}\max(s,0)$ to the price path. Confirm that bad news drops shocked prices below baseline by twice as much as equivalent good news lifts them.
> * __Score text with Claude and aggregate to a daily news factor:__ Generate plausible headlines per planted sentiment bucket via Claude, score each headline back to a number, and aggregate the per-item scores into a $T \times K$ daily news-factor matrix. Compare Claude's recovered scores to the planted ground truth so the LLM scoring noise is visible.
> * __Estimate $\nu_i$, deploy the extended engine, and quantify in-sample overfitting:__ Fit the SIM-with-news regression $g_{i,t} = \alpha_i + \beta_i g_{m,t} + \nu_i \,\text{news}_{i,t} + \varepsilon_{i,t}$ on a training window, run the news-extended engine on the test window, and contrast it with the SIM-only baseline. Then re-fit the loadings on the test window itself and watch the apparent alpha inflate — the in-sample-overfit version of the look-ahead pathology that plagues real-world news backtests.

Let's dive in!
___

## Setup, Data and Prerequisites
We begin by loading the `eCornellAIFinance` package and session dependencies via `Include.jl`. The package now exports the news-pipeline functions `simulate_news_corpus`, `generate_news_text!`, `score_news_with_claude!`, `aggregate_news_factor`, and `estimate_sim_with_news`, all introduced for this example.

In [ ]:
include("Include.jl");

### Implementations
Notebook-local helpers for the wealth scorecard and the per-bucket accuracy table.

`compute_scorecard(w_dict)` takes a `Dict{String,Vector{Float64}}` keyed by strategy label and returns a `DataFrame` with one row per strategy: terminal wealth as a multiple of initial, max drawdown from running peak, and a simple Sharpe ratio. `bucket_accuracy(corpus)` returns a `DataFrame` of mean Claude error per planted sentiment bucket so the scoring noise is visible per regime.

In [ ]:
"""
    compute_scorecard(w_dict::Dict{String,Vector{Float64}}) -> DataFrame

Per-strategy summary stats. Each value is a wealth trajectory; the scorecard
reports terminal wealth as a multiple of initial, max drawdown from running
peak, and a simple Sharpe ratio (total return / annualized daily-return vol).
"""
function compute_scorecard(w_dict::Dict{String,Vector{Float64}})::DataFrame
    rows = NamedTuple[];
    for (label, w) in w_dict
        returns  = diff(w) ./ w[1:end-1];
        peak     = accumulate(max, w);
        max_dd   = maximum((peak .- w) ./ peak);
        vol      = std(returns) * sqrt(252);
        mean_ret = w[end] / w[1] - 1.0;
        sharpe   = vol > 0 ? mean_ret / vol : 0.0;
        push!(rows, (strategy = label,
            terminal_multiple = w[end] / w[1],
            max_drawdown = max_dd,
            sharpe = sharpe));
    end
    return DataFrame(rows);
end;

"""
    bucket_accuracy(corpus::MyNewsCorpus) -> DataFrame

Mean signed Claude error per planted sentiment bucket. Buckets follow the
five-tone schema used by `_sentiment_bucket` in the package source so the
table aligns with the live-mode generation prompts.
"""
function bucket_accuracy(corpus::MyNewsCorpus)::DataFrame
    bucket_label(s) = s <= -0.6 ? "strongly negative" :
                      s <= -0.2 ? "mildly negative" :
                      s <  0.2  ? "neutral" :
                      s <  0.6  ? "mildly positive" : "strongly positive";
    rows = NamedTuple[];
    by_bucket = Dict{String,Vector{Float64}}();
    for it in corpus.items
        b = bucket_label(it.true_score);
        push!(get!(by_bucket, b, Float64[]), it.claude_score - it.true_score);
    end
    order = ["strongly negative", "mildly negative", "neutral", "mildly positive", "strongly positive"];
    for b in order
        haskey(by_bucket, b) || continue;
        errs = by_bucket[b];
        push!(rows, (bucket = b, n_items = length(errs),
            mean_error = mean(errs),
            mean_abs_error = mean(abs.(errs))));
    end
    return DataFrame(rows);
end;

### Constants
Global configuration: ticker universe, train and test windows, news scenario, and the engine offset. Constants live here so every Task references the same fixtures and the train and test boundaries are reproducible.

In [ ]:
# News scenario: kappa_neg = 2 * kappa_pos enforces the Hong-Lim-Stein asymmetry
# (bad news produces roughly twice the price reaction of equivalent good news).
NEWS_SCENARIO = build(MyNewsScenario, (label = "baseline_asymmetric",
    kappa_pos = 0.050, kappa_neg = 0.100,
    arrival_intensity = 0.80,
    sentiment_mean = 0.15, sentiment_sd = 0.45));

# Five tickers spanning sectors. The Setup loads the cached 2025 S&P data for these.
TICKERS = ["AAPL", "MSFT", "JPM", "WMT", "XOM"];
MARKET_TICKER = "SPY";

# Reproducibility seeds for the two stochastic stages.
CORPUS_SEED = 4242;          # Poisson arrivals + Gaussian sentiment draws
CLAUDE_NOISE_SEED = 7777;     # stub-mode Claude scoring noise
CLAUDE_NOISE_SD = 0.10;       # stub-mode Gaussian standard deviation on claude_score

# Engine constants.
B0 = 100_000.0;
EPSILON = 0.10;
OFFSET = 21;                  # warmup days before the engine takes its first allocation

We generate a fully synthetic price universe via `generate_hybrid_scenario`: the JumpHMM market surrogate produces the SPY-like market path, and per-ticker SIM (with the R²-preserving and marginal-preserving volatility correction documented in `MySIMCalibration`) composes ticker prices with planted $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$. Synthetic baseline keeps news and prices internally consistent, since real-2025 events do not bleed through. The bound names below are `prices_baseline::Array{Float64,2}` (a `T × K` ticker price matrix), `market_baseline::Array{Float64,1}` (the market index path), `T::Int`, and `K::Int`.

In [ ]:
prices_baseline, market_baseline, T, K = let
    # Hybrid-SIM scenario: JumpHMM market surrogate + per-ticker SIM with the
    # R²-preserving / marginal-preserving volatility correction baked into
    # generate_hybrid_scenario. One path so the engine A/B is deterministic.
    market_model = MyMarketSurrogateModel();
    portfolio    = MyPortfolioSurrogateModel();
    calib        = MySIMCalibration();

    scen = generate_hybrid_scenario(market_model, portfolio, calib, TICKERS;
        n_paths = 1, n_steps = 252, seed = 2024);

    P = scen.price_paths[1, :, :];     # T × K ticker prices
    M = scen.market_paths[1, :];       # T market index path
    P, M, scen.n_steps, length(TICKERS);
end;

println("Generated hybrid-SIM synthetic universe: T = ", T, " trading days, K = ", K, " tickers");

___

## Task 1: Generate the news corpus with asymmetric, forward-lagged price feedback
In this task, we generate a synthetic news corpus on top of the baseline 2025 prices and apply the asymmetric Hong-Lim-Stein shock **one day after publication** to produce a shocked price path. The shock is

$$\Delta_{i,t+1} = -\kappa_{\mathrm{neg}}\max(-s,0) + \kappa_{\mathrm{pos}}\max(s,0)$$

summed across items mentioning ticker $i$ on day $t$, with $\kappa_{\mathrm{neg}} = 2\kappa_{\mathrm{pos}}$ encoded directly in `NEWS_SCENARIO`. The shock applies with a one-day forward lag via `shock_lag = 1`, which mirrors the event-study convention where news after the close moves the following session's price. That lag is what makes $\text{news}_{i,t}$ genuinely predictive of $g_{i,t+1}$ and gives the news-extended $\gamma$ real out-of-sample alpha to chase.

In the code block below, we call `simulate_news_corpus` on the baseline price matrix, then unpack the corpus, the daily news factor, and the shocked prices. The bound names are `corpus::MyNewsCorpus`, `news_factor_true::Array{Float64,2}` (the $T \times K$ matrix aggregated from `true_score`), and `prices_shocked::Array{Float64,2}` (the news-shocked price matrix the engine will see).

In [ ]:
corpus, news_factor_true, prices_shocked = let
    # shock_lag = 1: news published on day t drives the return realized on day t+1,
    # mirroring the typical earnings-after-close-then-price-reacts-tomorrow event study.
    c = simulate_news_corpus(prices_baseline, TICKERS, NEWS_SCENARIO;
        seed = CORPUS_SEED, shock_lag = 1);
    c, c.news_factor, c.shocked_prices;
end;

println("Generated ", length(corpus.items), " news items across ", T, " trading days and ", K, " tickers");
println("Mean items per ticker per day: ", round(length(corpus.items) / (T * K), digits = 3));

The code block below plots the baseline price path against the shocked path for one ticker (the first in `TICKERS`) so the asymmetric drift is visible. We display in scaled units $W/W_0$ so the two paths share an axis regardless of starting price.

In [ ]:
let
    i = 1;  # first ticker
    base_scaled    = prices_baseline[:, i] ./ prices_baseline[1, i];
    shocked_scaled = prices_shocked[:, i] ./ prices_shocked[1, i];
    plot(1:T, base_scaled, label = "baseline", color = :black, linewidth = 2,
        xlabel = "trading day", ylabel = "price / price[1]",
        title = "$(TICKERS[i]): baseline vs news-shocked (asymmetric κ)",
        legend = :topleft);
    plot!(1:T, shocked_scaled, label = "shocked", color = :red, linewidth = 2);
end

___

## Task 2: Score the corpus with Claude
In this task, we generate Claude-written headlines for each item (live mode) or use the seeded stub text (cached mode), then ask Claude to score each headline back to a number in $[-1, +1]$. The round trip introduces realistic LLM scoring noise so the downstream regression sees a degraded version of the planted signal. The cached mode uses a Gaussian noise model `claude_score = clamp(true_score + N(0, σ_{stub}), -1, +1)` with `σ_{stub} = 0.10` so the notebook runs end-to-end without an API key.

In the code block below, we run the two-stage pipeline (text generation, then scoring) in cached mode. Both functions accept `live = true` and an `api_key` keyword once an Anthropic key is wired in via `ENV["ANTHROPIC_API_KEY"]`. The bound name is `news_factor_claude::Array{Float64,2}`, the daily news factor aggregated from Claude's recovered scores rather than the planted ground truth.

In [ ]:
news_factor_claude = let
    # text generation: cached mode is a no-op (stub text from simulate_news_corpus)
    generate_news_text!(corpus; live = false);

    # scoring: cached mode adds Gaussian noise around true_score
    score_news_with_claude!(corpus; live = false,
        cached_noise_sd = CLAUDE_NOISE_SD, seed = CLAUDE_NOISE_SEED);

    # re-aggregate the daily factor from Claude's scores rather than ground truth
    aggregate_news_factor(corpus.items, T, TICKERS; use_score = :claude_score);
end;

println("First 3 items, true vs Claude score:");
for it in corpus.items[1:3]
    println("  ", it.ticker, " day ", it.publication_day,
        "  true=", round(it.true_score, digits = 3),
        "  claude=", round(it.claude_score, digits = 3));
end

The code block below summarizes the scoring noise per planted sentiment bucket. The `mean_error` column is the mean signed difference `claude_score - true_score`; the `mean_abs_error` column is the mean magnitude. With `σ_{stub} = 0.10` we expect mean signed error near zero and mean absolute error near $\sqrt{2/\pi} \cdot 0.10 \approx 0.08$.

In [ ]:
let
    df = bucket_accuracy(corpus);
    df.mean_error = round.(df.mean_error; digits = 4);
    df.mean_abs_error = round.(df.mean_abs_error; digits = 4);
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end

___

## Task 3: Estimate $\nu_i$, deploy the extended $\gamma$, and demonstrate in-sample overfitting
In this task, we estimate the news-extended SIM model $g_{i,t} = \alpha_i + \beta_i g_{m,t} + \nu_i\,\text{news}_{i,t} + \varepsilon_{i,t}$ on a training window, deploy the news-extended $\gamma$ in the rebalancing engine on a held-out test window, and compare it to the SIM-only baseline. We then expose **in-sample overfitting** by re-fitting $\nu_i$ on the test window itself and observing the apparent alpha inflate. Real-world news backtests usually call this look-ahead bias because the parameter window leaks future-dated headlines into past-dated trades; with synthetic data and no real timestamps, parameter overfitting is the cleaner label for the same pathology.

We use a 50/50 train and test split of the synthetic year. The training window is days $1, \ldots, T_{\mathrm{train}}$; the test window is days $T_{\mathrm{train}}+1, \ldots, T$. The engine's first allocation falls on day $T_{\mathrm{train}} + \text{OFFSET}$.

In the code block below, we compute realized growth rates from the shocked price matrix, slice to the training window, and fit the SIM-with-news regression. The bound names are `T_train::Int`, `sim_params_train::Dict{String,Tuple{Float64,Float64,Float64}}` (the recovered $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$ per ticker), and `nu_loadings_train::Array{Float64,1}` (the recovered $\nu_i$ per ticker, in `TICKERS` order).

In [ ]:
T_train, sim_params_train, nu_loadings_train = let
    T_train_local = T ÷ 2;

    # CCGR growth rates (length T-1) from shocked prices and SPY
    growth = log.(prices_shocked[2:end, :] ./ prices_shocked[1:end-1, :]);
    gm     = log.(market_baseline[2:end] ./ market_baseline[1:end-1]);

    # Predictive alignment: growth[k] (realized on day k+1) is driven by news[k]
    # (published on day k). We pair growth[k] with news_factor_claude[k, :].
    news_aligned = news_factor_claude[1:end-1, :];

    train_idx = 1:(T_train_local - 1);
    (sp, nu) = estimate_sim_with_news(
        growth[train_idx, :], gm[train_idx], news_aligned[train_idx, :], TICKERS);
    T_train_local, sp, nu;
end;

println("T_train = ", T_train, "  T_test = ", T - T_train);

The code block below displays the recovered $(\alpha_i, \beta_i, \nu_i)$ per ticker. Tickers with $|\nu_i|$ well above zero are news-sensitive in the synthetic DGP; tickers with $\nu_i \approx 0$ are insensitive. Because the planted shock is symmetric across tickers (same scenario for all), the variation in $\hat\nu_i$ reflects sampling noise plus the LLM scoring noise we layered on in Task 2.

In [ ]:
let
    rows = NamedTuple[];
    for (i, t) in enumerate(TICKERS)
        (a, b, se) = sim_params_train[t];
        push!(rows, (ticker = t,
            α = round(a; digits = 5),
            β = round(b; digits = 4),
            ν = round(nu_loadings_train[i]; digits = 5),
            σ_ε = round(se; digits = 5)));
    end
    df = DataFrame(rows);
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end

With $(\alpha_i, \beta_i, \nu_i)$ in hand, we run the rebalancing engine on the test window for two strategies. The baseline strategy ignores news (`nu_loadings = zeros(K)`); the extended strategy passes the recovered $\nu_i$ alongside the daily Claude news factor. Both engines see the same shocked prices, the same EMA-derived $\lambda_t$, and the same trigger rules, so any wealth difference attributes to the news term in $\gamma$.

In the code block below, we build the rebalancing context, compute $\lambda_t$ via EMA crossover on SPY, and run the engine twice over the test window. The bound names are `wealth_baseline::Array{Float64,1}`, `wealth_extended::Array{Float64,1}` (per-day wealth trajectories on the test window), and `lambda_series::Array{Float64,1}` (the $\lambda_t$ shared by both runs).

In [ ]:
wealth_baseline, wealth_extended, lambda_series = let
    Δt = 1.0 / 252.0;

    # market growth + EMA λ shared across both engine runs
    gm_raw   = compute_market_growth(market_baseline; Δt = Δt);
    gm_ema   = compute_ema(gm_raw; window = 10);
    ema_s    = compute_ema(market_baseline; window = 21);
    ema_l    = compute_ema(market_baseline; window = 63);
    λ_series = compute_lambda(ema_s, ema_l; G = 10.0);

    # engine on day t reads yesterday's news: shift news_factor rows forward by 1
    news_paths_trade = vcat(zeros(1, K), news_factor_claude[1:end-1, :]);

    # marketdata format expected by the engine: column 1 = day index, then K price columns
    pmatrix = hcat(collect(1:T), prices_shocked);

    # context anchored at T_train so the engine's first allocation falls in the test window
    test_steps = T - (T_train + OFFSET);
    rules = build(MyTriggerRules, (
        max_drawdown = 0.30, max_turnover = 1.0,
        rebalance_schedule = ones(Int, test_steps)));
    ctx = build(MyRebalancingContextModel, (
        B = B0, tickers = TICKERS, marketdata = pmatrix,
        marketfactor = gm_ema, sim_parameters = sim_params_train,
        lambda = 0.0, Δt = Δt, epsilon = EPSILON));

    # baseline (no news) vs extended (news + ν loadings)
    res_baseline = run_rebalancing_engine(ctx, rules, λ_series;
        offset = T_train + OFFSET);
    res_extended = run_rebalancing_engine(ctx, rules, λ_series;
        offset = T_train + OFFSET,
        news_paths = news_paths_trade, nu_loadings = nu_loadings_train);

    # per-day wealth trajectory: shares · prices + cash
    function wealth_trajectory(res, prices, offset_local, n_steps)
        w = zeros(n_steps + 1);
        for d in 0:n_steps
            r = res[d];
            actual_day = offset_local + d;
            w[d + 1] = r.cash + sum(r.shares[i] * prices[actual_day, i] for i in 1:K);
        end
        return w;
    end
    wb = wealth_trajectory(res_baseline, prices_shocked, T_train + OFFSET, test_steps);
    we = wealth_trajectory(res_extended, prices_shocked, T_train + OFFSET, test_steps);
    wb, we, λ_series;
end;

println("test wealth multiples — baseline: ", round(wealth_baseline[end] / B0, digits = 3),
    "  extended: ", round(wealth_extended[end] / B0, digits = 3));

The code block below computes the scorecard for both strategies on the test window, then plots the wealth trajectories in scaled units $W/W_0$.

In [ ]:
let
    df = compute_scorecard(Dict(
        "SIM only (baseline)" => wealth_baseline,
        "SIM + news (extended)" => wealth_extended));
    df.terminal_multiple = round.(df.terminal_multiple; digits = 4);
    df.max_drawdown = round.(df.max_drawdown; digits = 4);
    df.sharpe = round.(df.sharpe; digits = 4);
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end

In [ ]:
let
    n = length(wealth_baseline);
    plot(1:n, wealth_baseline ./ B0,
        label = "SIM only", color = :black, linewidth = 2,
        xlabel = "test-window trading day", ylabel = "W / W₀",
        title = "Test-window wealth: SIM-only vs news-extended γ", legend = :topleft);
    plot!(1:n, wealth_extended ./ B0,
        label = "SIM + news", color = :red, linewidth = 2);
    # mark terminal points with open red circles per the course convention
    scatter!([n], [wealth_baseline[end] / B0], color = :white, markerstrokecolor = :red, markersize = 7, label = "");
    scatter!([n], [wealth_extended[end] / B0], color = :white, markerstrokecolor = :red, markersize = 7, label = "");
end

___
### In-sample overfitting of $\nu$
We now repeat the regression on the **test window itself**, deploy the resulting $\nu_i$ in the engine over the same test window, and compare to the clean prior-window result. Because the test data is what the engine trades on, fitting the loadings to that data lets every ticker's $\nu_i$ absorb the realized news-driven returns. The engine then "predicts" what it has already seen, and the apparent alpha is mostly that absorption. The clean version trains $\nu_i$ strictly on the prior window so the test trajectory is the engine's honest out-of-sample alpha. Real-world news backtests dress this same overfitting in temporal clothing (a parameter fit on data later than the trade timestamp) and call it look-ahead bias; with synthetic data and no real timestamps, parameter overfitting is the more honest label.

In the code block below, we re-fit on the test window and re-run the engine, keeping the train-window $\alpha_i$ and $\beta_i$ untouched so the gap is attributable purely to the leaked $\nu_i$. The bound names are `nu_loadings_leaked::Array{Float64,1}` and `wealth_leaked::Array{Float64,1}`.

In [ ]:
nu_loadings_leaked, wealth_leaked = let
    Δt = 1.0 / 252.0;
    growth = log.(prices_shocked[2:end, :] ./ prices_shocked[1:end-1, :]);
    gm     = log.(market_baseline[2:end] ./ market_baseline[1:end-1]);
    news_aligned = news_factor_claude[1:end-1, :];    # predictive pairing

    # CHEATING: fit ONLY the news loadings on the test window the engine will
    # trade on. We keep the honest train-window α and β so the gap between
    # clean and leaked is attributable purely to look-ahead-fitted ν.
    test_idx = T_train:(T - 1);
    (_sp_unused, nu_leak) = estimate_sim_with_news(
        growth[test_idx, :], gm[test_idx], news_aligned[test_idx, :], TICKERS);

    # rebuild and re-run the engine using sim_params_train (clean) + nu_leak (LEAKED)
    gm_raw   = compute_market_growth(market_baseline; Δt = Δt);
    gm_ema   = compute_ema(gm_raw; window = 10);
    ema_s    = compute_ema(market_baseline; window = 21);
    ema_l    = compute_ema(market_baseline; window = 63);
    λ_series = compute_lambda(ema_s, ema_l; G = 10.0);
    pmatrix  = hcat(collect(1:T), prices_shocked);
    news_paths_trade = vcat(zeros(1, K), news_factor_claude[1:end-1, :]);
    test_steps = T - (T_train + OFFSET);
    rules = build(MyTriggerRules, (
        max_drawdown = 0.30, max_turnover = 1.0,
        rebalance_schedule = ones(Int, test_steps)));
    ctx = build(MyRebalancingContextModel, (
        B = B0, tickers = TICKERS, marketdata = pmatrix,
        marketfactor = gm_ema, sim_parameters = sim_params_train,
        lambda = 0.0, Δt = Δt, epsilon = EPSILON));
    res_leak = run_rebalancing_engine(ctx, rules, λ_series;
        offset = T_train + OFFSET,
        news_paths = news_paths_trade, nu_loadings = nu_leak);

    function wealth_trajectory(res, prices, offset_local, n_steps)
        w = zeros(n_steps + 1);
        for d in 0:n_steps
            r = res[d];
            actual_day = offset_local + d;
            w[d + 1] = r.cash + sum(r.shares[i] * prices[actual_day, i] for i in 1:K);
        end
        return w;
    end
    wl = wealth_trajectory(res_leak, prices_shocked, T_train + OFFSET, test_steps);
    nu_leak, wl;
end;

println("ν loadings: prior-window vs test-window-fitted (look-ahead)");
for (i, t) in enumerate(TICKERS)
    println("  ", t,
        "  ν_clean=", round(nu_loadings_train[i], digits = 4),
        "  ν_leaked=", round(nu_loadings_leaked[i], digits = 4));
end

The code block below tabulates the three strategies side by side. The clean extended result is the engine's honest out-of-sample alpha from news; the leaked result is what the same engine reports when the loadings absorb test-window returns directly. The gap between them is the look-ahead alpha, which would not survive deployment.

In [ ]:
let
    df = compute_scorecard(Dict(
        "SIM only (baseline)" => wealth_baseline,
        "SIM + news, prior-window ν (out-of-sample)" => wealth_extended,
        "SIM + news, test-window ν (in-sample fit)" => wealth_leaked));
    df.terminal_multiple = round.(df.terminal_multiple; digits = 4);
    df.max_drawdown = round.(df.max_drawdown; digits = 4);
    df.sharpe = round.(df.sharpe; digits = 4);
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end

In [ ]:
let
    n = length(wealth_baseline);
    plot(1:n, wealth_baseline ./ B0,
        label = "SIM only", color = :black, linewidth = 2,
        xlabel = "test-window trading day", ylabel = "W / W₀",
        title = "In-sample overfitting: out-of-sample ν vs in-sample-fit ν", legend = :topleft);
    plot!(1:n, wealth_extended ./ B0,
        label = "SIM + news (out-of-sample ν)", color = :red, linewidth = 2);
    plot!(1:n, wealth_leaked ./ B0,
        label = "SIM + news (in-sample ν)", color = :red, linestyle = :dash, linewidth = 2);
end

___

## Summary
We extended the Cobb-Douglas engine with a news-derived feature, ran it on a fully synthetic universe (JumpHMM market + per-ticker SIM with the volatility correction) with planted news shocks layered on top, and made look-ahead bias visible by re-fitting the loadings on the trading window itself.

> __Key Takeaways:__
>
> * __Asymmetric news feedback shapes the price path:__ The Hong-Lim-Stein shock $\Delta = -\kappa_{\mathrm{neg}}\max(-s,0) + \kappa_{\mathrm{pos}}\max(s,0)$ with $\kappa_{\mathrm{neg}} = 2\kappa_{\mathrm{pos}}$ drives shocked prices below baseline more than equivalent positive news lifts them. Generating news with this DGP gives Task 3's regression a real signal to recover.
> * __Claude scoring degrades but preserves the signal:__ Bucketing planted sentiment into five tones, asking Claude to write headlines, and asking Claude to read them back introduces realistic scoring noise without destroying the per-asset signal. The recovered $\nu_i$ are biased toward zero by the noise but stay close to the planted values.
> * __In-sample overfitting inflates apparent alpha:__ Fitting $\nu_i$ on the test window the engine trades on lets each loading absorb the realized news-driven returns. The in-sample fit outperforms the clean prior-window fit by a margin that vanishes when deployed forward; the prior-window result is the honest out-of-sample alpha. Real-world news backtests usually call this look-ahead bias because the parameter window leaks future-dated headlines into past-dated trades — synthetic data with no real timestamps makes parameter overfitting the cleaner label.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The news corpus used here is synthetic, generated by `simulate_news_corpus` from a planted sentiment distribution. The Claude scoring step runs in cached stub mode by default; live mode requires `ENV["ANTHROPIC_API_KEY"]` and incurs Anthropic API charges.
___